# Bab 3 — Eksperimen Statistik dan Pengujian Hipotesis

**Buku:** *Practical Statistics for Data Scientists* (O'Reilly)

## Ringkasan Bab
Eksperimen statistik (seperti A/B testing) dirancang untuk menguji signifikansi perbedaan antara kelompok perlakuan dan kontrol. Bab ini mencakup:

1. **A/B Testing** dan perancangan eksperimen.
2. **Uji Hipotesis** (H0 vs H1).
3. **Uji Permutasi (Resampling)**.
4. **Signifikansi Statistik dan P-value**.
5. **Uji-t, ANOVA, dan Chi-Square**.
6. **Multiple Testing dan Statistical Power**.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats import power
import matplotlib.pyplot as plt
import seaborn as sns

DATA = Path('..') / 'data'
sns.set_theme(style='whitegrid')
random.seed(1)
np.random.seed(1)

## 1. Uji Permutasi (Permutation Test)

**Teori.** Uji permutasi menggabungkan data kelompok, mengacaknya, dan mengevaluasi apakah perbedaan yang diamati signifikan atau hanya kebetulan semata di bawah hipotesis nol (H0).

**Dataset:** `web_page_data.csv`

In [ ]:
session_times = pd.read_csv(DATA / 'web_page_data.csv')
session_times['Time'] = 100 * session_times['Time']   # convert to seconds
session_times.groupby('Page')['Time'].agg(['count', 'mean'])

In [ ]:
ax = session_times.boxplot(by='Page', column='Time', figsize=(4, 4))
ax.set_xlabel(''); ax.set_ylabel('Time (in seconds)')
plt.suptitle(''); plt.title('')
plt.tight_layout(); plt.show()

In [ ]:
mean_a = session_times[session_times.Page == 'Page A'].Time.mean()
mean_b = session_times[session_times.Page == 'Page B'].Time.mean()
observed_diff = mean_b - mean_a
print('Observed difference (B - A):', round(observed_diff, 2), 'seconds')

def perm_fun(x, n_a, n_b):
    n = n_a + n_b
    idx_b = set(random.sample(range(n), n_b))
    idx_a = set(range(n)) - idx_b
    return x.loc[list(idx_b)].mean() - x.loc[list(idx_a)].mean()

n_a = (session_times.Page == 'Page A').sum()
n_b = (session_times.Page == 'Page B').sum()
perm_diffs = np.array([perm_fun(session_times.Time, n_a, n_b) for _ in range(1000)])

fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(perm_diffs, bins=11, rwidth=0.9)
ax.axvline(observed_diff, color='black', lw=2)
ax.text(observed_diff, 190, 'Observed\ndifference', bbox={'facecolor': 'white'})
ax.set_xlabel('Session time differences (seconds)'); ax.set_ylabel('Frequency')
plt.tight_layout(); plt.show()

print('Permutation p-value (one-sided):', np.mean(perm_diffs > observed_diff))

## 2. Signifikansi Statistik dan P-Value

**Teori.** P-value mengukur peluang memperoleh hasil se-ekstrem atau lebih ekstrem dari data yang diamati jika H0 benar.

In [ ]:
obs_pct_diff = 100 * (200 / 23739 - 182 / 22588)
print(f'Observed difference: {obs_pct_diff:.4f}%')

conversion = pd.Series([0] * 45945 + [1] * 382)
perm_diffs = [100 * perm_fun(conversion, 23739, 22588) for _ in range(1000)]
perm_diffs = np.array(perm_diffs)

fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(perm_diffs, bins=11, rwidth=0.9)
ax.axvline(obs_pct_diff, color='black', lw=2)
ax.set_xlabel('Conversion rate difference (percentage points)'); ax.set_ylabel('Frequency')
plt.tight_layout(); plt.show()

print('p-value (one-sided):', np.mean(perm_diffs > obs_pct_diff))

In [ ]:
table = np.array([[200, 23739 - 200], [182, 22588 - 182]])
chi2, p_value, dof, expected = stats.chi2_contingency(table)
print(f'Chi-square p-value (one-sided approx.): {p_value / 2:.4f}')

## 3. Uji-t dan ANOVA

**Teori.** Uji-t dan ANOVA adalah padanan analitis berbasis rumus untuk membandingkan mean antar kelompok.

In [ ]:
a = session_times[session_times.Page == 'Page A'].Time
b = session_times[session_times.Page == 'Page B'].Time
res = stats.ttest_ind(a, b, equal_var=False)
print(f't-statistic: {res.statistic:.3f}')
print(f'p-value (one-sided): {res.pvalue / 2:.4f}')

In [ ]:
four_sessions = pd.read_csv(DATA / 'four_sessions.csv')
ax = four_sessions.boxplot(by='Page', column='Time', figsize=(4, 4))
ax.set_xlabel('Page'); ax.set_ylabel('Time (seconds)')
plt.suptitle(''); plt.title('')
plt.tight_layout(); plt.show()

In [ ]:
observed_variance = four_sessions.groupby('Page')['Time'].mean().var()
print('Observed variance among page means:', observed_variance)

def perm_test(df):
    df = df.copy()
    df['Time'] = np.random.permutation(df['Time'].values)
    return df.groupby('Page')['Time'].mean().var()

perm_variance = [perm_test(four_sessions) for _ in range(3000)]
print('Permutation p-value:', np.mean([v > observed_variance for v in perm_variance]))

fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(perm_variance, bins=11, rwidth=0.9)
ax.axvline(observed_variance, color='black', lw=2)
ax.set_xlabel('Variance of page means'); ax.set_ylabel('Frequency')
plt.tight_layout(); plt.show()

In [ ]:
model = smf.ols('Time ~ Page', data=four_sessions).fit()
print(sm.stats.anova_lm(model))

## 4. Uji Chi-Square dan Statistical Power

**Teori.** Uji chi-square menguji kesesuaian frekuensi data kategorikal, sementara analisis power menentukan ukuran sampel yang dibutuhkan untuk mendeteksi efek tertentu.

In [ ]:
click_rate = pd.read_csv(DATA / 'click_rates.csv')
clicks = click_rate.pivot(index='Click', columns='Headline', values='Rate')
print(clicks)

In [ ]:
box = [1] * 34 + [0] * 2966
expected_clicks = 34 / 3
expected = [expected_clicks, 1000 - expected_clicks]

def chi2(observed, expected):
    return np.sum([[(o - e) ** 2 / e for o in row] for row, e in zip(observed, expected)])

chi2_observed = chi2(clicks.values, expected)

def perm_chi2():
    random.shuffle(box)
    sample_clicks = [sum(box[0:1000]), sum(box[1000:2000]), sum(box[2000:3000])]
    sample_noclicks = [1000 - n for n in sample_clicks]
    return chi2([sample_clicks, sample_noclicks], expected)

perm_results = [perm_chi2() for _ in range(2000)]
print(f'Observed chi2: {chi2_observed:.4f}')
print('Resampled p-value:', np.mean(np.array(perm_results) > chi2_observed))

chisq, pvalue, dof, exp = stats.chi2_contingency(clicks)
print(f'scipy chi2: {chisq:.4f}, p-value: {pvalue:.4f}, dof: {dof}')

In [ ]:
imanishi = pd.read_csv(DATA / 'imanishi_data.csv')
imanishi.columns = ['Digit', 'Frequency']
print(imanishi.T)
observed = imanishi['Frequency'].values
res = stats.chisquare(observed)
print(f'chi2 = {res.statistic:.2f}, p-value = {res.pvalue:.2e}')

In [ ]:
effect_size = sm.stats.proportion_effectsize(0.0121, 0.011)
analysis = power.NormalIndPower()
n = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.8, alternative='larger')
print(f'Sample size needed per group: {n:,.0f}')